# 03 - CharCNN + MLP mini-batch

Notebook generado automáticamente para el proyecto de predicción de década.

## 0. Instalación y configuración

In [ ]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install pandas numpy scipy scikit-learn matplotlib tqdm

In [ ]:
# =========================
# Imports generales
# =========================

import os
import gc
import math
import time
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

# =========================
# Visualización
# =========================

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# =========================
# Scikit-learn
# =========================

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    classification_report,
    confusion_matrix,
)

# =========================
# PyTorch
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [ ]:
# =========================
# Configuración general para GPU
# =========================

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("CONFIGURACIÓN DEL ENTORNO")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    print("Memoria asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
    print("Memoria reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

print("=" * 70)


In [ ]:
# =========================
# Limpieza segura de memoria
# =========================

def clean_memory(verbose=True):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        if verbose:
            print("Memoria asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
            print("Memoria reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

clean_memory()


## 1. Configuración

In [ ]:
# =========================
# Configuración del experimento: CharCNN + MLP mini-batch
# =========================

# =========================
# Configuración común para modelos de caracteres
# =========================

TRAIN_PATH = "train.csv"
TEXT_COL = "text"
LABEL_COL = "decade"

MAX_CHAR_LEN = 1400
RANDOM_CROP_TRAIN = True

EPOCHS = 30
PATIENCE = 2

BATCH_SIZE = 256 if DEVICE.type == "cuda" else 64

LR = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

DROPOUT = 0.35

print("Batch size:", BATCH_SIZE)
print("Max char length:", MAX_CHAR_LEN)


CHAR_EMB_DIM = 160
CNN_CHANNELS = 192
CNN_KERNELS = [3, 5, 7, 9, 11]
RES_CHANNELS = CNN_CHANNELS * len(CNN_KERNELS)
MLP_HIDDEN_1 = 1024
MLP_HIDDEN_2 = 512

CHECKPOINT_DIR = "checkpoints_03_charcnn_mlp_minibatch"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best.pt")
LAST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "last.pt")

CONFIG = {
    "MAX_CHAR_LEN": MAX_CHAR_LEN,
    "VOCAB_SIZE": None,
    "NUM_CLASSES": None,
    "CHAR_EMB_DIM": CHAR_EMB_DIM,
    "CNN_CHANNELS": CNN_CHANNELS,
    "CNN_KERNELS": CNN_KERNELS,
    "MLP_HIDDEN_1": MLP_HIDDEN_1,
    "MLP_HIDDEN_2": MLP_HIDDEN_2,
    "DROPOUT": DROPOUT,
}

print("Checkpoints:", CHECKPOINT_DIR)


## 2. Datos

In [ ]:
# =========================
# Carga de datos y mapeo de etiquetas
# =========================

df = pd.read_csv(TRAIN_PATH)

assert TEXT_COL in df.columns, f"No existe columna '{TEXT_COL}'"
assert LABEL_COL in df.columns, f"No existe columna '{LABEL_COL}'"

df = df[[TEXT_COL, LABEL_COL]].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

decades = sorted(df[LABEL_COL].unique())
decade_to_idx = {decade: idx for idx, decade in enumerate(decades)}
idx_to_decade = {idx: decade for decade, idx in decade_to_idx.items()}

df["label_idx"] = df[LABEL_COL].map(decade_to_idx)

NUM_CLASSES = len(decades)

print("Shape:", df.shape)
print("Número de clases:", NUM_CLASSES)
print("Década mínima:", min(decades))
print("Década máxima:", max(decades))
print("Clases:", decades)

# Chequeos explícitos de etiquetas extremas
assert min(decades) == 150, "La década mínima no es 150. Revisa el dataset."
assert max(decades) == 188, "La década máxima no es 188. Revisa el dataset."
assert NUM_CLASSES == 39, f"Se esperaban 39 clases, pero hay {NUM_CLASSES}."

display(df.head())
display(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# =========================
# Split train / validation / test
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# Verificar que las clases extremas estén en todos los splits
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    present = set(part[LABEL_COL].unique())
    print(name, "contiene 150:", 150 in present, "| contiene 188:", 188 in present)


In [ ]:
# =========================
# Vocabulario de caracteres y datasets
# =========================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

char_counter = Counter()
for text in train_df[TEXT_COL].astype(str):
    char_counter.update(text)

chars = [PAD_TOKEN, UNK_TOKEN] + sorted(char_counter.keys())
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

PAD_IDX = char_to_idx[PAD_TOKEN]
UNK_IDX = char_to_idx[UNK_TOKEN]
VOCAB_SIZE = len(char_to_idx)

print("Vocab size:", VOCAB_SIZE)
print("Caracteres más frecuentes:", char_counter.most_common(20))

def crop_text_train(text, max_len=MAX_CHAR_LEN):
    text = str(text)
    if not RANDOM_CROP_TRAIN or len(text) <= max_len:
        return text[:max_len]
    start = random.randint(0, len(text) - max_len)
    return text[start:start + max_len]

def crop_text_eval(text, max_len=MAX_CHAR_LEN, mode="center"):
    text = str(text)
    if len(text) <= max_len:
        return text
    if mode == "start":
        return text[:max_len]
    if mode == "end":
        return text[-max_len:]
    start = (len(text) - max_len) // 2
    return text[start:start + max_len]

def get_eval_crops(text, max_len=MAX_CHAR_LEN):
    text = str(text)
    if len(text) <= max_len:
        return [text]
    crops = [
        text[:max_len],
        text[(len(text)-max_len)//2:(len(text)-max_len)//2 + max_len],
        text[-max_len:],
    ]
    out, seen = [], set()
    for c in crops:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

def encode_chars(text, max_len=MAX_CHAR_LEN):
    text = str(text)[:max_len]
    ids = [char_to_idx.get(ch, UNK_IDX) for ch in text]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    return ids[:max_len]

class CharDataset(Dataset):
    def __init__(self, dataframe, train_mode=False):
        self.texts = dataframe[TEXT_COL].astype(str).tolist()
        self.labels = dataframe["label_idx"].astype(int).tolist()
        self.train_mode = train_mode

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.train_mode:
            text = crop_text_train(text)
        else:
            text = crop_text_eval(text, mode="center")
        return {
            "text": text,
            "char_ids": torch.tensor(encode_chars(text), dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

def char_collate_fn(batch):
    return {
        "texts": [x["text"] for x in batch],
        "char_ids": torch.stack([x["char_ids"] for x in batch]),
        "labels": torch.stack([x["label"] for x in batch])
    }

train_dataset = CharDataset(train_df, train_mode=True)
val_dataset = CharDataset(val_df, train_mode=False)
test_dataset = CharDataset(test_df, train_mode=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=char_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=char_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=char_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 3. Modelo CNN

In [ ]:
# =========================
# Modelo CharCNN residual + MLP
# =========================

class ResidualConvBlock(nn.Module):
    def __init__(self, channels, kernel_size=5, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size // 2) * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        x = F.gelu(x + residual)
        return x

class CharCNNMLPClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, CHAR_EMB_DIM, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(0.15)

        self.convs = nn.ModuleList([
            nn.Conv1d(CHAR_EMB_DIM, CNN_CHANNELS, kernel_size=k, padding=k//2)
            for k in CNN_KERNELS
        ])

        conv_dim = CNN_CHANNELS * len(CNN_KERNELS)
        self.bn = nn.BatchNorm1d(conv_dim)

        self.res_blocks = nn.Sequential(
            ResidualConvBlock(conv_dim, kernel_size=5, dilation=1, dropout=0.20),
            ResidualConvBlock(conv_dim, kernel_size=5, dilation=2, dropout=0.20),
            ResidualConvBlock(conv_dim, kernel_size=5, dilation=4, dropout=0.20),
        )

        pooled_dim = conv_dim * 2

        self.classifier = nn.Sequential(
            nn.Linear(pooled_dim, MLP_HIDDEN_1),
            nn.LayerNorm(MLP_HIDDEN_1),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN_1, MLP_HIDDEN_2),
            nn.LayerNorm(MLP_HIDDEN_2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(MLP_HIDDEN_2, num_classes),
        )

    def forward(self, char_ids):
        x = self.embedding(char_ids)
        x = self.emb_dropout(x)
        x = x.transpose(1, 2)

        x = torch.cat([F.gelu(conv(x)) for conv in self.convs], dim=1)
        x = self.bn(x)
        x = self.res_blocks(x)

        max_pool = torch.max(x, dim=2).values
        avg_pool = torch.mean(x, dim=2)
        x = torch.cat([max_pool, avg_pool], dim=1)

        return self.classifier(x)

CONFIG["VOCAB_SIZE"] = VOCAB_SIZE
CONFIG["NUM_CLASSES"] = NUM_CLASSES

model = CharCNNMLPClassifier(VOCAB_SIZE, NUM_CLASSES).to(DEVICE)
print(model)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:
MODEL_CLASS = CharCNNMLPClassifier

In [ ]:
# =========================
# Métricas, entrenamiento y predicción
# =========================

idx_to_decade_tensor = torch.tensor([idx_to_decade[i] for i in range(NUM_CLASSES)], dtype=torch.long, device=DEVICE)

def compute_metrics_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    pred_decades = idx_to_decade_tensor[preds]
    true_decades = idx_to_decade_tensor[labels]
    abs_error = torch.abs(pred_decades - true_decades)
    return {
        "acc": (preds == labels).float().mean().item(),
        "acc_pm1": (abs_error <= 1).float().mean().item(),
        "acc_pm2": (abs_error <= 2).float().mean().item(),
        "mae": abs_error.float().mean().item(),
    }

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_logits, all_labels = [], []

    for batch in tqdm(loader, desc="Training", leave=False):
        char_ids = batch["char_ids"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = model(char_ids)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        all_logits.append(logits.detach())
        all_labels.append(labels.detach())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    metrics = compute_metrics_from_logits(all_logits, all_labels)
    metrics["loss"] = total_loss / total_examples
    return metrics

@torch.no_grad()
def evaluate(model, loader, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_logits, all_labels = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        char_ids = batch["char_ids"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = model(char_ids)
            loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        all_logits.append(logits.detach())
        all_labels.append(labels.detach())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    metrics = compute_metrics_from_logits(all_logits, all_labels)
    metrics["loss"] = total_loss / total_examples
    return metrics

@torch.no_grad()
def predict_text_multicrop(model, text):
    model.eval()
    crops = get_eval_crops(text)
    batch = torch.stack([torch.tensor(encode_chars(c), dtype=torch.long) for c in crops]).to(DEVICE)
    with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
        logits = model(batch)
        probs = torch.softmax(logits, dim=1)
    avg_probs = probs.mean(dim=0)
    pred_idx = int(torch.argmax(avg_probs).item())
    return pred_idx

@torch.no_grad()
def predict_dataframe_multicrop(model, dataframe):
    preds = []
    for text in tqdm(dataframe[TEXT_COL].astype(str).tolist(), desc="Predicting"):
        preds.append(predict_text_multicrop(model, text))
    return np.array(preds)


## 4. Entrenamiento

In [ ]:
# =========================
# Loop principal: prioriza accuracy
# =========================

history = []
best_val_acc = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 70)

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader, desc="Validation")
    scheduler.step(val_metrics["acc"])

    row = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_pm1": train_metrics["acc_pm1"],
        "train_pm2": train_metrics["acc_pm2"],
        "train_mae": train_metrics["mae"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_pm1": val_metrics["acc_pm1"],
        "val_pm2": val_metrics["acc_pm2"],
        "val_mae": val_metrics["mae"],
    }
    history.append(row)

    print(f"Train loss {row['train_loss']:.4f} | acc {row['train_acc']:.4f} | MAE {row['train_mae']:.3f}")
    print(f"Val   loss {row['val_loss']:.4f} | acc {row['val_acc']:.4f} | MAE {row['val_mae']:.3f} | ±1 {row['val_pm1']:.4f}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best_val_acc": best_val_acc,
        "char_to_idx": char_to_idx,
        "idx_to_char": idx_to_char,
        "decade_to_idx": decade_to_idx,
        "idx_to_decade": idx_to_decade,
        "decades": decades,
        "config": CONFIG,
    }, LAST_MODEL_PATH)

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best_val_acc": best_val_acc,
            "best_val_metrics": val_metrics,
            "char_to_idx": char_to_idx,
            "idx_to_char": idx_to_char,
            "decade_to_idx": decade_to_idx,
            "idx_to_decade": idx_to_decade,
            "decades": decades,
            "config": CONFIG,
        }, BEST_MODEL_PATH)

        print("Nuevo mejor checkpoint guardado.")
    else:
        epochs_without_improvement += 1
        print(f"Sin mejora por {epochs_without_improvement} época(s).")

    clean_memory(verbose=False)

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping activado.")
        break

print("Mejor época:", best_epoch)
print("Mejor val accuracy:", best_val_acc)


## 5. Evaluación, matriz de confusión y submission normal

In [ ]:
# =========================
# Curvas, test, matriz de confusión y submission del modelo con validación
# =========================

history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(10, 4))
plt.plot(history_df["epoch"], history_df["train_acc"], label="Train acc")
plt.plot(history_df["epoch"], history_df["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

checkpoint = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Mejor época:", checkpoint["epoch"])
print("Best val acc:", checkpoint["best_val_acc"])

test_metrics = evaluate(model, test_loader, desc="Test")
print("TEST:", test_metrics)

# Matriz de confusión con multi-crop en test
test_pred_idx = predict_dataframe_multicrop(model, test_df)
y_true = test_df["label_idx"].values
y_pred = test_pred_idx

print("Accuracy test multicrop:", accuracy_score(y_true, y_pred))
print("MAE test multicrop:", mean_absolute_error([idx_to_decade[i] for i in y_true], [idx_to_decade[i] for i in y_pred]))

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_df = pd.DataFrame(cm, index=[idx_to_decade[i] for i in range(NUM_CLASSES)], columns=[idx_to_decade[i] for i in range(NUM_CLASSES)])
display(cm_df)

plt.figure(figsize=(15, 12))
plt.imshow(cm, aspect="auto")
plt.title("Matriz de confusión sin normalizar")
plt.xlabel("Década predicha")
plt.ylabel("Década real")
plt.xticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)], rotation=90)
plt.yticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)])
plt.colorbar(label="Conteo")
plt.tight_layout()
plt.show()

print("Predicciones por clase en test:")
display(pd.Series([idx_to_decade[i] for i in y_pred]).value_counts().sort_index())

# Submission con modelo de validación
EVAL_PATH = "eval.csv"
SUBMISSION_PATH = "submission_validation_model.csv"

eval_df = pd.read_csv(EVAL_PATH)
assert "id" in eval_df.columns and "text" in eval_df.columns
eval_df["text"] = eval_df["text"].fillna("").astype(str)

eval_pred_idx = predict_dataframe_multicrop(model, eval_df)
answers = [idx_to_decade[int(i)] for i in eval_pred_idx]

submission = pd.DataFrame({"id": eval_df["id"].values, "answer": answers})
submission.to_csv(SUBMISSION_PATH, index=False)

print("Submission guardada:", SUBMISSION_PATH)
display(submission.head())
print("Predicciones por década en eval:")
display(submission["answer"].value_counts().sort_index())


## 6. Entrenamiento full-data y submission final

In [ ]:
# =========================
# Entrenamiento final full-data y submission
# =========================

FINAL_EPOCHS = int(checkpoint["epoch"])
FINAL_CHECKPOINT_DIR = CHECKPOINT_DIR + "_final"
os.makedirs(FINAL_CHECKPOINT_DIR, exist_ok=True)
FINAL_MODEL_PATH = os.path.join(FINAL_CHECKPOINT_DIR, "final_full_data.pt")

print("Entrenando modelo final con todo train.csv por", FINAL_EPOCHS, "épocas.")

# reconstruir vocabulario con todo el dataset
char_counter = Counter()
for text in df[TEXT_COL].astype(str):
    char_counter.update(text)

chars = [PAD_TOKEN, UNK_TOKEN] + sorted(char_counter.keys())
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}
PAD_IDX = char_to_idx[PAD_TOKEN]
UNK_IDX = char_to_idx[UNK_TOKEN]
VOCAB_SIZE = len(char_to_idx)

full_df = df.copy().reset_index(drop=True)
final_dataset = CharDataset(full_df, train_mode=True)
final_loader = DataLoader(final_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=char_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

try:
    model.to("cpu")
    del model
except Exception:
    pass
clean_memory()

final_model = MODEL_CLASS(VOCAB_SIZE, NUM_CLASSES).to(DEVICE)
final_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=max(FINAL_EPOCHS, 1))
final_scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type=="cuda"))

def train_final_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for batch in tqdm(loader, desc="Final training", leave=False):
        char_ids = batch["char_ids"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        final_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = model(char_ids)
            loss = final_criterion(logits, labels)

        final_scaler.scale(loss).backward()
        final_scaler.unscale_(final_optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        final_scaler.step(final_optimizer)
        final_scaler.update()

        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)
    return total_loss / total_examples

final_history = []
for epoch in range(1, FINAL_EPOCHS + 1):
    print(f"Final epoch {epoch}/{FINAL_EPOCHS}")
    loss = train_final_epoch(final_model, final_loader)
    final_scheduler.step()
    final_history.append({"epoch": epoch, "loss": loss, "lr": final_optimizer.param_groups[0]["lr"]})
    print("loss:", loss)
    clean_memory(verbose=False)

torch.save({
    "epoch": FINAL_EPOCHS,
    "model_state_dict": final_model.state_dict(),
    "history": final_history,
    "char_to_idx": char_to_idx,
    "idx_to_char": idx_to_char,
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
    "config": CONFIG,
}, FINAL_MODEL_PATH)

print("Modelo final guardado:", FINAL_MODEL_PATH)

FINAL_SUBMISSION_PATH = "submission_full_data_model.csv"
eval_df = pd.read_csv(EVAL_PATH)
eval_df["text"] = eval_df["text"].fillna("").astype(str)

eval_pred_idx = predict_dataframe_multicrop(final_model, eval_df)
answers = [idx_to_decade[int(i)] for i in eval_pred_idx]

submission_final = pd.DataFrame({"id": eval_df["id"].values, "answer": answers})
submission_final.to_csv(FINAL_SUBMISSION_PATH, index=False)

print("Submission full-data guardada:", FINAL_SUBMISSION_PATH)
display(submission_final.head())
display(submission_final["answer"].value_counts().sort_index())
